# LLE–PPS surrogate statistic test — 60-s Processed PPG

Reusable session pipeline. Each included 60-s Processed window receives exactly 39 deterministic PPS realizations. Rosenstein LLE is computed with the original window's frozen configuration; surrogate `fit_r2` is retained as a diagnostic and is not an invalidation criterion. The pipeline writes one window-level summary CSV and one long-form surrogate CSV per session.


In [1]:
# Cell 1 — Imports, paths, frozen configuration, and output schemas
import hashlib
import json
import os
import re
import sys
import time
from importlib import import_module
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed, parallel_config


def find_repo_root(start):
    for path in (start, *start.parents):
        if (path / "phase1" / "src").is_dir():
            return path
    raise FileNotFoundError("Repository root was not found.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

loader = import_module("phase1.src.dataloader.loader")
pps = import_module("phase1.src.surrogates.pps")
lyapunov = import_module("phase1.src.chaos.lyapunov")
load_segmented_session = loader.load_segmented_session
generate_pps_signal = pps.generate_pps_signal
compute_rosenstein_lle = lyapunov.compute_rosenstein_lle

REPRESENTATION = "processed"
WINDOW_SIZE_S = 60
M = 39
ALPHA = 0.05
ALTERNATIVE = "two-sided"
MASTER_SEED = 20260828
MIN_INITIAL_PAIRS = 50
MIN_FIT_PAIRS = 30
ORIGINAL_MIN_R2 = 0.90
SURROGATE_MIN_R2 = 0.0  # fit_r2 remains diagnostic only for PPS data.
N_JOBS = min(8, os.cpu_count() or 1)
STATE_NAMES = {0: "Awake", 1: "Drowsy"}
WINDOW_KEYS = ["session", "window_id", "state", "window_size_s", "representation"]

SEGMENTED_DATA_DIR = REPO_ROOT / "phase1" / "segmentated_data" / "dhdata"
LLE_INPUT_DIR = REPO_ROOT / "phase1" / "results" / "lle" / "processed"
RHO_INPUT_DIR = REPO_ROOT / "phase1" / "results" / "pps" / "pps_radius_calibration" / "csv"
OUTPUT_DIR = REPO_ROOT / "phase1" / "results" / "lle" / "surrogates"

SUMMARY_COLUMNS = [
    "session", "window_id", "state", "window_size_s", "representation",
    "sampling_rate_hz", "n_samples",
    "m", "tau_s", "tau_samples",
    "theiler_s", "theiler_samples", "fit_start_s", "fit_end_s", "max_follow_s",
    "original_lle", "original_fit_r2", "original_valid", "original_qc_reason",
    "n_surrogates_expected", "n_surrogates_valid",
    "surrogate_lle_mean", "surrogate_lle_std", "surrogate_lle_median",
    "surrogate_lle_q25", "surrogate_lle_q75",
    "surrogate_lle_min", "surrogate_lle_max",
    "lle_gap_mean", "lle_gap_median", "z_score_surrogate",
    "rank_ascending", "rank_direction", "p_two_sided", "reject_alpha_0_05",
    "rho_star", "master_seed",
]
SURROGATE_COLUMNS = [
    "session", "window_id", "state", "window_size_s", "representation",
    "surrogate_id", "surrogate_seed", "rho_star",
    "sampling_rate_hz", "m", "tau_s", "tau_samples",
    "theiler_s", "theiler_samples", "fit_start_s", "fit_end_s", "max_follow_s",
    "surrogate_lle", "surrogate_fit_r2",
    "n_embedded", "n_pairs_initial", "n_pairs_fit_min",
    "valid", "qc_reason",
]

session_pattern = re.compile(r"session_(\d+)_processed_rosenstein_lle[.]csv$")
AVAILABLE_SESSION_IDS = sorted(
    int(match.group(1))
    for path in LLE_INPUT_DIR.glob("session_*_processed_rosenstein_lle.csv")
    if (match := session_pattern.match(path.name))
)
assert len(AVAILABLE_SESSION_IDS) == 20
assert M == 39 and ALTERNATIVE == "two-sided" and np.isclose(2 / (M + 1), ALPHA)
print(f"Available sessions ({len(AVAILABLE_SESSION_IDS)}): {AVAILABLE_SESSION_IDS}")
print(f"Frozen scope: Processed, {WINDOW_SIZE_S} s, M={M}; output={OUTPUT_DIR}")


Available sessions (20): [1, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 21, 22, 23, 25]
Frozen scope: Processed, 60 s, M=39; output=/home/vutu0809/Desktop/NTSA_Foundation/phase1/results/lle/surrogates


In [2]:
# Cell 2 — Deterministic seeds and the frozen two-sided +1 rank rule
def derive_window_seed(identity, master_seed=MASTER_SEED):
    payload = json.dumps(
        {"master_seed": int(master_seed), **identity},
        sort_keys=True, separators=(",", ":"),
    ).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "little")


def derive_surrogate_seeds(window_seed, count=M):
    return [
        int.from_bytes(
            hashlib.sha256(f"{window_seed}:{surrogate_id}".encode("ascii")).digest()[:8],
            "little",
        )
        for surrogate_id in range(1, count + 1)
    ]


def two_sided_rank_test(original_value, surrogate_values, alpha=ALPHA):
    original = float(original_value)
    values = np.asarray(surrogate_values, dtype=float)
    if not np.isfinite(original):
        raise ValueError("Original LLE must be finite.")
    if values.ndim != 1 or values.size != M or not np.all(np.isfinite(values)):
        raise ValueError(f"Rank test requires exactly {M} finite surrogate LLE values.")
    n_lower = int(np.count_nonzero(values <= original))
    n_upper = int(np.count_nonzero(values >= original))
    n_equal = int(np.count_nonzero(values == original))
    n_less = int(np.count_nonzero(values < original))
    denominator = values.size + 1
    p_lower = (n_lower + 1) / denominator
    p_upper = (n_upper + 1) / denominator
    p_value = min(1.0, 2.0 * min(p_lower, p_upper))
    reject = bool(p_value <= alpha)
    direction = "none"
    if reject and p_upper < p_lower:
        direction = "higher"
    elif reject and p_lower < p_upper:
        direction = "lower"
    return {
        "rank_ascending": float(1 + n_less + 0.5 * n_equal),
        "rank_direction": direction,
        "p_two_sided": float(p_value),
        "reject_alpha_0_05": reject,
    }

print("Deterministic seed derivation and rank test are ready.")


Deterministic seed derivation and rank test are ready.


In [3]:
# Cell 3 — Load and fail-fast match one session's original windows, LLE rows, and rho rows
def load_session_tasks(session_id):
    session_id = int(session_id)
    if session_id not in AVAILABLE_SESSION_IDS:
        raise ValueError(f"Unknown session_id={session_id}; available={AVAILABLE_SESSION_IDS}")

    lle_path = LLE_INPUT_DIR / f"session_{session_id:02d}_processed_rosenstein_lle.csv"
    rho_path = RHO_INPUT_DIR / f"pps_radius_session_{session_id:02d}.csv"
    if not lle_path.is_file() or not rho_path.is_file():
        raise FileNotFoundError(f"Missing LLE or rho input for session {session_id}.")

    lle_rows = pd.read_csv(lle_path)
    lle_rows = lle_rows.loc[
        lle_rows["session"].eq(session_id)
        & lle_rows["representation"].str.casefold().eq(REPRESENTATION)
        & lle_rows["window_size_s"].eq(WINDOW_SIZE_S)
        & lle_rows["analysis_included"].astype(bool)
    ].copy()
    rho_rows = pd.read_csv(rho_path)
    rho_rows = rho_rows.loc[
        rho_rows["session"].eq(session_id)
        & rho_rows["representation"].str.casefold().eq(REPRESENTATION)
        & rho_rows["window_size"].eq(WINDOW_SIZE_S)
    ].copy()
    if lle_rows.empty:
        raise ValueError(f"Session {session_id} has no included 60-s Processed windows.")

    batches, metadata = load_segmented_session(
        f"sample_{session_id}.csv", data_dir=SEGMENTED_DATA_DIR,
        window_sizes=WINDOW_SIZE_S, representation=REPRESENTATION,
        stationarity_only=True,
    )
    batch = batches[WINDOW_SIZE_S]
    tasks = []
    for index, window_id in enumerate(batch["window_id"]):
        state = STATE_NAMES[int(batch["label"][index])]
        identity = {
            "session": session_id, "window_id": int(window_id),
            "state": state, "window_size_s": WINDOW_SIZE_S,
            "representation": REPRESENTATION,
        }
        lle_match = lle_rows.loc[
            lle_rows["window_id"].eq(window_id)
            & lle_rows["state"].str.casefold().eq(state.casefold())
        ]
        rho_match = rho_rows.loc[
            rho_rows["window_id"].eq(window_id)
            & rho_rows["state"].str.casefold().eq(state.casefold())
        ]
        if len(lle_match) != 1 or len(rho_match) != 1:
            raise ValueError(f"Non-unique LLE/rho match for {identity}.")
        reference = lle_match.iloc[0].to_dict()
        rho_reference = rho_match.iloc[0]
        signal = np.asarray(batch["signal"][index], dtype=float)
        fs = float(batch["fs"])
        if signal.ndim != 1 or not np.all(np.isfinite(signal)):
            raise ValueError(f"Invalid signal for {identity}.")
        checks = [
            signal.size == int(reference["n_samples"]),
            np.isclose(fs, float(reference["sampling_rate_hz"]), rtol=0, atol=1e-9),
            int(reference["tau_samples"]) == int(rho_reference["tau_samples"]),
            signal.size == int(rho_reference["N"]),
            np.isclose(fs, float(rho_reference["sampling_rate"]), rtol=0, atol=1e-9),
            bool(reference["signal_finite"]),
        ]
        if not all(checks):
            raise ValueError(f"Original metadata mismatch for {identity}.")
        tasks.append({
            **identity, "signal": signal, "sampling_rate_hz": fs,
            "rho_star": float(rho_reference["rho_star"]),
            "reference": reference,
        })

    task_identities = pd.DataFrame([{key: task[key] for key in WINDOW_KEYS} for task in tasks])
    lle_identities = lle_rows[WINDOW_KEYS].copy()
    if task_identities.duplicated(WINDOW_KEYS).any() or lle_identities.duplicated(WINDOW_KEYS).any():
        raise ValueError("Duplicated original window identity.")
    if set(map(tuple, task_identities.itertuples(index=False, name=None))) != set(
        map(tuple, lle_identities.itertuples(index=False, name=None))
    ):
        raise ValueError("Segmented and LLE cohorts do not match exactly.")
    if len(tasks) != len(rho_rows):
        raise ValueError("Included-window and rho row counts differ.")
    return tasks

print("Session loader and metadata matching are ready.")


Session loader and metadata matching are ready.


In [4]:
# Cell 4 — Evaluate one original window and its 39 PPS realizations
def evaluate_window(task):
    identity = {key: task[key] for key in WINDOW_KEYS}
    reference = task["reference"]
    common_config = {
        "sampling_rate": task["sampling_rate_hz"],
        "m": int(reference["m"]),
        "tau_samples": int(reference["tau_samples"]),
        "fit_start_s": float(reference["fit_start_s"]),
        "fit_end_s": float(reference["fit_end_s"]),
        "max_follow_s": float(reference["max_follow_s"]),
        "theiler_s": float(reference["theiler_s"]),
        "min_initial_pairs": MIN_INITIAL_PAIRS,
        "min_fit_pairs": MIN_FIT_PAIRS,
    }
    original_result = compute_rosenstein_lle(
        task["signal"], **common_config, min_r2=ORIGINAL_MIN_R2
    )
    original_checks = [
        np.isfinite(original_result.lle),
        np.isclose(original_result.lle, float(reference["lle_1_per_s"]), rtol=0, atol=1e-12),
        np.isclose(original_result.fit_r2, float(reference["fit_r2"]), rtol=0, atol=1e-12),
        original_result.n_embedded == int(reference["n_embedded"]),
        original_result.n_pairs_initial == int(reference["n_pairs_initial"]),
        original_result.n_pairs_fit_min == int(reference["n_pairs_fit_min"]),
        original_result.valid == bool(reference["valid"]),
        original_result.qc_reason == str(reference["qc_reason"]),
    ]
    if not all(original_checks):
        raise RuntimeError(f"Original frozen-LLE mismatch for {identity}.")

    window_seed = derive_window_seed(identity)
    surrogate_rows = []
    for surrogate_id, surrogate_seed in enumerate(derive_surrogate_seeds(window_seed), start=1):
        surrogate_signal = generate_pps_signal(
            task["signal"], tau=common_config["tau_samples"],
            m=common_config["m"], rho=task["rho_star"],
            rng=np.random.default_rng(surrogate_seed), return_indices=False,
        )
        result = compute_rosenstein_lle(
            surrogate_signal, **common_config, min_r2=SURROGATE_MIN_R2
        )
        if not np.isfinite(result.lle):
            raise RuntimeError(f"Non-finite surrogate LLE for {identity}, id={surrogate_id}.")
        surrogate_rows.append({
            **identity, "surrogate_id": surrogate_id,
            "surrogate_seed": surrogate_seed, "rho_star": task["rho_star"],
            "sampling_rate_hz": task["sampling_rate_hz"],
            "m": common_config["m"], "tau_s": float(reference["tau_s"]),
            "tau_samples": common_config["tau_samples"],
            "theiler_s": float(reference["theiler_s"]),
            "theiler_samples": original_result.theiler_samples,
            "fit_start_s": common_config["fit_start_s"],
            "fit_end_s": common_config["fit_end_s"],
            "max_follow_s": common_config["max_follow_s"],
            "surrogate_lle": result.lle, "surrogate_fit_r2": result.fit_r2,
            "n_embedded": result.n_embedded,
            "n_pairs_initial": result.n_pairs_initial,
            "n_pairs_fit_min": result.n_pairs_fit_min,
            "valid": result.valid, "qc_reason": result.qc_reason,
        })

    surrogate_frame = pd.DataFrame(surrogate_rows, columns=SURROGATE_COLUMNS)
    values = surrogate_frame["surrogate_lle"].to_numpy(dtype=float)
    if len(values) != M or not surrogate_frame["valid"].all():
        raise RuntimeError(f"Incomplete valid surrogate population for {identity}.")
    surrogate_mean = float(np.mean(values))
    surrogate_std = float(np.std(values, ddof=1))
    if not np.isfinite(surrogate_std) or surrogate_std <= 0:
        raise RuntimeError(f"Invalid surrogate LLE standard deviation for {identity}.")
    q25, q75 = np.quantile(values, [0.25, 0.75])
    rank = two_sided_rank_test(original_result.lle, values)
    summary_row = {
        **identity, "sampling_rate_hz": task["sampling_rate_hz"],
        "n_samples": task["signal"].size, "m": common_config["m"],
        "tau_s": float(reference["tau_s"]), "tau_samples": common_config["tau_samples"],
        "theiler_s": float(reference["theiler_s"]),
        "theiler_samples": original_result.theiler_samples,
        "fit_start_s": common_config["fit_start_s"],
        "fit_end_s": common_config["fit_end_s"],
        "max_follow_s": common_config["max_follow_s"],
        "original_lle": original_result.lle, "original_fit_r2": original_result.fit_r2,
        "original_valid": original_result.valid, "original_qc_reason": original_result.qc_reason,
        "n_surrogates_expected": M, "n_surrogates_valid": int(surrogate_frame["valid"].sum()),
        "surrogate_lle_mean": surrogate_mean, "surrogate_lle_std": surrogate_std,
        "surrogate_lle_median": float(np.median(values)),
        "surrogate_lle_q25": float(q25), "surrogate_lle_q75": float(q75),
        "surrogate_lle_min": float(np.min(values)), "surrogate_lle_max": float(np.max(values)),
        "lle_gap_mean": original_result.lle - surrogate_mean,
        "lle_gap_median": original_result.lle - float(np.median(values)),
        "z_score_surrogate": (original_result.lle - surrogate_mean) / surrogate_std,
        **rank, "rho_star": task["rho_star"], "master_seed": MASTER_SEED,
    }
    return summary_row, surrogate_rows

print("Per-window PPS and LLE evaluator is ready.")


Per-window PPS and LLE evaluator is ready.


In [5]:
# Cell 5 — Output validation and reusable session runner
def validate_output_frames(summary, surrogates, tasks):
    if list(summary.columns) != SUMMARY_COLUMNS:
        raise ValueError("Summary schema/order mismatch.")
    if list(surrogates.columns) != SURROGATE_COLUMNS:
        raise ValueError("Surrogate schema/order mismatch.")
    if len(summary) != len(tasks) or len(surrogates) != len(tasks) * M:
        raise ValueError("Unexpected summary or surrogate row count.")
    if summary.duplicated(WINDOW_KEYS).any():
        raise ValueError("Duplicated window-level summary identity.")
    surrogate_keys = WINDOW_KEYS + ["surrogate_id"]
    if surrogates.duplicated(surrogate_keys).any():
        raise ValueError("Duplicated surrogate_id within a window.")

    expected_ids = set(range(1, M + 1))
    groups = surrogates.groupby(WINDOW_KEYS, sort=False)
    if len(groups) != len(tasks):
        raise ValueError("Surrogate population has missing/extra windows.")
    for identity, group in groups:
        if len(group) != M or set(group["surrogate_id"].astype(int)) != expected_ids:
            raise ValueError(f"Surrogate IDs are not exactly 1..{M} for {identity}.")
    if not np.all(np.isfinite(surrogates["surrogate_lle"])):
        raise ValueError("Non-finite surrogate LLE slope detected.")
    if not surrogates["valid"].astype(bool).all():
        raise ValueError("A surrogate failed non-R² Rosenstein QC.")
    if not summary["n_surrogates_expected"].eq(M).all():
        raise ValueError("n_surrogates_expected mismatch.")
    valid_counts = groups["valid"].sum().rename("expected_valid").reset_index()
    valid_check = summary.merge(valid_counts, on=WINDOW_KEYS, validate="one_to_one")
    if not valid_check["n_surrogates_valid"].eq(valid_check["expected_valid"]).all():
        raise ValueError("n_surrogates_valid mismatch.")

    expected = pd.DataFrame([{
        **{key: task[key] for key in WINDOW_KEYS},
        "expected_rho": task["rho_star"],
        "expected_original": float(task["reference"]["lle_1_per_s"]),
    } for task in tasks])
    summary_check = summary.merge(expected, on=WINDOW_KEYS, validate="one_to_one")
    if not np.allclose(summary_check["rho_star"], summary_check["expected_rho"], rtol=0, atol=1e-12):
        raise ValueError("Summary rho_star mismatch.")
    if not np.allclose(summary_check["original_lle"], summary_check["expected_original"], rtol=0, atol=1e-12):
        raise ValueError("Summary original LLE mismatch.")
    surrogate_check = surrogates.merge(expected[WINDOW_KEYS + ["expected_rho"]], on=WINDOW_KEYS, validate="many_to_one")
    if not np.allclose(surrogate_check["rho_star"], surrogate_check["expected_rho"], rtol=0, atol=1e-12):
        raise ValueError("Surrogate rho_star mismatch.")

    for _, row in summary.iterrows():
        mask = np.logical_and.reduce([surrogates[key].eq(row[key]) for key in WINDOW_KEYS])
        values = surrogates.loc[mask, "surrogate_lle"].to_numpy(dtype=float)
        test = two_sided_rank_test(float(row["original_lle"]), values)
        checks = [
            np.isclose(row["rank_ascending"], test["rank_ascending"]),
            row["rank_direction"] == test["rank_direction"],
            np.isclose(row["p_two_sided"], test["p_two_sided"]),
            bool(row["reject_alpha_0_05"]) == test["reject_alpha_0_05"],
            np.isclose(row["lle_gap_mean"], row["original_lle"] - np.mean(values)),
            np.isclose(row["lle_gap_median"], row["original_lle"] - np.median(values)),
        ]
        if not all(checks):
            raise ValueError(f"Rank/deviation summary mismatch for window_id={row['window_id']}.")
    return True


def run_statistic_test_lle(session_id, *, overwrite=False, n_jobs=N_JOBS):
    session_id = int(session_id)
    tasks = load_session_tasks(session_id)
    summary_path = OUTPUT_DIR / f"session_{session_id:02d}_lle_pps_summary.csv"
    surrogate_path = OUTPUT_DIR / f"session_{session_id:02d}_lle_pps_surrogates.csv"

    if summary_path.exists() or surrogate_path.exists():
        if not (summary_path.exists() and surrogate_path.exists()):
            raise FileExistsError("Only one output exists; use overwrite=True after auditing it.")
        if not overwrite:
            summary = pd.read_csv(summary_path)
            surrogates = pd.read_csv(surrogate_path)
            validate_output_frames(summary, surrogates, tasks)
            print(f"Session {session_id}: output đã tồn tại và hợp lệ — bỏ qua.")
            return summary, surrogates

    started = time.perf_counter()
    print(f"Session {session_id}: bắt đầu {len(tasks)} windows.")
    with parallel_config(backend="loky", inner_max_num_threads=1):
        result_stream = Parallel(
            n_jobs=n_jobs, verbose=0, return_as="generator_unordered"
        )(
            delayed(evaluate_window)(task) for task in tasks
        )
        bundles = []
        for completed, bundle in enumerate(result_stream, start=1):
            bundles.append(bundle)
            print(f"Session {session_id}: window {completed}/{len(tasks)}")
    summary = pd.DataFrame([bundle[0] for bundle in bundles], columns=SUMMARY_COLUMNS)
    surrogates = pd.DataFrame(
        [row for bundle in bundles for row in bundle[1]], columns=SURROGATE_COLUMNS
    )
    summary = summary.sort_values(WINDOW_KEYS).reset_index(drop=True)
    surrogates = surrogates.sort_values(WINDOW_KEYS + ["surrogate_id"]).reset_index(drop=True)
    validate_output_frames(summary, surrogates, tasks)

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    summary_tmp = summary_path.with_suffix(".csv.tmp")
    surrogate_tmp = surrogate_path.with_suffix(".csv.tmp")
    summary.to_csv(summary_tmp, index=False)
    surrogates.to_csv(surrogate_tmp, index=False)
    summary_tmp.replace(summary_path)
    surrogate_tmp.replace(surrogate_path)

    # Read back and validate serialization before returning.
    saved_summary = pd.read_csv(summary_path)
    saved_surrogates = pd.read_csv(surrogate_path)
    validate_output_frames(saved_summary, saved_surrogates, tasks)
    elapsed_min = (time.perf_counter() - started) / 60
    print(f"Session {session_id}: hoàn tất trong {elapsed_min:.2f} phút.")
    return saved_summary, saved_surrogates

print("run_statistic_test_lle(session_id) is ready.")


run_statistic_test_lle(session_id) is ready.


In [7]:
# Cell 7 — Run/validate all real dataset sessions
# Session 1 is already saved, so it will only be validated and skipped.
SESSIONS = [1, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 17, 18, 19, 21, 22, 23, 25]
assert SESSIONS == AVAILABLE_SESSION_IDS

for session_id in SESSIONS:
    run_statistic_test_lle(session_id, overwrite=False, n_jobs=N_JOBS)


Session 23: window 29/62
Session 23: window 30/62
Session 23: window 31/62
Session 23: window 32/62
Session 23: window 33/62
Session 23: window 34/62
Session 23: window 35/62
Session 23: window 36/62
Session 23: window 37/62
Session 23: window 38/62
Session 23: window 39/62
Session 23: window 40/62
Session 23: window 41/62
Session 23: window 42/62
Session 23: window 43/62
Session 23: window 44/62
Session 23: window 45/62
Session 23: window 46/62
Session 23: window 47/62
Session 23: window 48/62
Session 23: window 49/62
Session 23: window 50/62
Session 23: window 51/62
Session 23: window 52/62
Session 23: window 53/62
Session 23: window 54/62
Session 23: window 55/62
Session 23: window 56/62
Session 23: window 57/62
Session 23: window 58/62
Session 23: window 59/62
Session 23: window 60/62
Session 23: window 61/62
Session 23: window 62/62
Session 23: hoàn tất trong 1.10 phút.
Session 25: bắt đầu 42 windows.
Session 25: window 1/42
Session 25: window 2/42
Session 25: window 3/42
Session 